In [1]:
# =================================================================================
# CHEATGRASS DATA PROCESSING PIPELINE (refactored & robust)
# - Stable, unique global_id
# - "cheatgrass" naming (no VEDU)
# - Diagnostics + experimental set generation
#
# NEW (at bottom):
# - build_validmask_dataset(): clone existing *_data.npy / *_mask.npy into a
#   new directory tree and add *_valid.npy (valid / observed mask).
# =================================================================================

import os
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", message="The get_cmap function was deprecated", category=DeprecationWarning)
warnings.filterwarnings("ignore", "GeoSeries.notna", UserWarning)

# ----------------------------- CONFIG -------------------------------------------
DEFAULT_INPUT_PATH = "/home/rbielski/SAL_Git_Projects/Cheatgrass/filtered_cheatgrass_cleaned_valid.geojson"
INPUT_PATH = os.getenv("CHEATGRASS_INPUT_PATH", DEFAULT_INPUT_PATH)
print(f"Input dataset: {INPUT_PATH} (override with CHEATGRASS_INPUT_PATH)")

PERCENT_VARIATIONS = [50, 75, 90, 100]
AREA_VARIATIONS = [900, 8100]  # m² thresholds
TARGET_CRS = "EPSG:32611"

# Treat <= this as "missing" InfestSqM so we can fallback to geometry area
MIN_GEOFALLBACK_AREA = 1.0

CHEATGRASS_ALIASES = {
    "CHEATGRASS",
    "DOWNY BROME",
    "BROMUS TECTORUM",
    "B. TECTORUM",
    "BROMUS  TECTORUM",
}

def _normalize_sciname(s: str) -> str:
    s = str(s).upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)  # drop punctuation
    s = re.sub(r"\s+", " ", s).strip()  # squeeze spaces
    return s

CHEATGRASS_ALIASES_NORMALIZED = {_normalize_sciname(a) for a in CHEATGRASS_ALIASES}


# ----------------------------- ID HELPERS ---------------------------------------
def _sanitize_token(s: str) -> str:
    s = re.sub(r'[^A-Za-z0-9_-]+', '-', s)
    s = s.strip('-')
    return s or 'X'

def _uniqify_series(s: pd.Series) -> pd.Series:
    counts = {}
    out = []
    for v in s.astype(str):
        if v in counts:
            counts[v] += 1
            out.append(f"{v}-{counts[v]:02d}")
        else:
            counts[v] = 0
            out.append(v)
    return pd.Series(out, index=s.index)

def _build_gid(gdf: "gpd.GeoDataFrame") -> pd.Series:
    # start from existing column if present
    if "global_id" in gdf.columns:
        gid = gdf["global_id"].astype("string")
    else:
        gid = pd.Series([pd.NA] * len(gdf), index=gdf.index, dtype="string")

    # normalize and null-out null-like strings
    gid = gid.str.strip()
    gid = gid.where(~gid.str.lower().isin(["", "none", "nan", "null"]), pd.NA)

    # fallback 1: any common ID-ish column
    for cand in ["OBJECTID", "objectid", "ObjectID", "FID", "fid"]:
        if cand in gdf.columns:
            gid = gid.fillna(gdf[cand].astype("string"))
            break

    # fallback 2: stable row-based id
    row_ids = gdf.index.to_series().map(lambda i: f"ROW-{int(i):06d}")
    gid = gid.fillna(row_ids.astype("string"))

    # sanitize and ensure uniqueness
    gid = gid.map(_sanitize_token)
    gid = _uniqify_series(gid)

    return gid.astype("string")


# ----------------------------- LOAD & PREP --------------------------------------
def load_and_prepare_cheatgrass(filepath: str) -> gpd.GeoDataFrame:
    print("\n--- Loading Cleaned Cheatgrass Dataset ---")
    gdf = gpd.read_file(filepath)
    print(f"✅ Loaded {len(gdf)} rows | {len(gdf.columns)} columns")

    # Species name normalized
    if "SciName" in gdf.columns:
        gdf["primary_sp"] = gdf["SciName"].apply(_normalize_sciname)
    else:
        gdf["primary_sp"] = "UNKNOWN"

    # % cover
    if "primary_sp_percent" not in gdf.columns:
        if "Percentcov_num" in gdf.columns:
            gdf["primary_sp_percent"] = gdf["Percentcov_num"].fillna(0)
        elif "Percentcov" in gdf.columns:
            gdf["primary_sp_percent"] = pd.to_numeric(
                gdf["Percentcov"], errors="coerce"
            ).fillna(0)
        else:
            gdf["primary_sp_percent"] = 0.0

    # Area in m² from acres (if provided)
    if "InfestSqM" not in gdf.columns:
        if "InfestAcre_clean" in gdf.columns:
            gdf["InfestSqM"] = gdf["InfestAcre_clean"].fillna(0) * 4046.86
        else:
            gdf["InfestSqM"] = 0.0

    # Cheatgrass presence flag
    if "is_cheatgrass" not in gdf.columns:
        gdf["is_cheatgrass"] = gdf["primary_sp"].apply(
            lambda x: 1 if any(a in x for a in CHEATGRASS_ALIASES_NORMALIZED) else 0
        )
    gdf["is_cheatgrass"] = gdf["is_cheatgrass"].fillna(0).astype(int)

    # Valid geometries only
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

    # Always (re)build robust global_id
    gdf["global_id"] = _build_gid(gdf)

    # Diagnostics
    print(f"✅ Geometry-valid rows: {len(gdf)}")
    print(
        "Alias match positives (is_cheatgrass==1):",
        int((gdf["is_cheatgrass"] == 1).sum()),
        "/",
        len(gdf),
    )
    preview_cols = [
        c
        for c in [
            "global_id",
            "primary_sp",
            "primary_sp_percent",
            "InfestSqM",
            "is_cheatgrass",
        ]
        if c in gdf.columns
    ]
    print(gdf[preview_cols].head())
    return gdf


def _diagnostic_counts(df: pd.DataFrame, label: str):
    if df.empty:
        return
    print(
        f"[{label}] class balance is_cheatgrass ->",
        df["is_cheatgrass"].value_counts().to_dict(),
    )
    if "primary_sp_percent" in df.columns:
        print(
            f"[{label}] primary_sp_percent stats -> "
            f"min={df.primary_sp_percent.min()} max={df.primary_sp_percent.max()}"
        )
    if "InfestSqM" in df.columns:
        print(
            f"[{label}] InfestSqM summary -> "
            f"count>0={(df.InfestSqM > 0).sum()} "
            f"zeros/NaN={(df.InfestSqM.isna() | (df.InfestSqM <= 0)).sum()}"
        )


# ----------------------- EXPERIMENTAL SET GENERATION ----------------------------
def generate_experimental_datasets(
    filepath: str,
    percent_thresholds: list,
    area_thresholds: list,
    target_crs: str = TARGET_CRS,
    include_controls: bool = False,  # default False (your file appears to be all cheatgrass)
    test_size: float = 0.2,
    random_state: int = 42,
):
    base_gdf = load_and_prepare_cheatgrass(filepath)
    _diagnostic_counts(base_gdf, "BASE")

    # Split (fallback to random if only one class or tiny dataset)
    if base_gdf["is_cheatgrass"].nunique() < 2 or len(base_gdf) < 4:
        print(
            "⚠ Not enough class diversity or too few rows for stratified split; using random split."
        )
        master_train_gdf, final_test_set = train_test_split(
            base_gdf, test_size=min(0.5, test_size), random_state=random_state
        )
    else:
        master_train_gdf, final_test_set = train_test_split(
            base_gdf,
            test_size=test_size,
            stratify=base_gdf["is_cheatgrass"],
            random_state=random_state,
        )

    print("\n--- Master Sets Created ---")
    print(
        f"Master Training Set: {len(master_train_gdf)} | Final Test Set: {len(final_test_set)}"
    )
    _diagnostic_counts(master_train_gdf, "TRAIN")

    # Reproject + robust area fallback
    reprojected = master_train_gdf.to_crs(target_crs)
    reprojected["geom_area_sqm"] = reprojected.geometry.area

    if "InfestSqM" in reprojected.columns:
        # use InfestSqM unless missing/too small, otherwise geometry area
        reprojected["area_sqm"] = reprojected["InfestSqM"].where(
            ~(
                reprojected["InfestSqM"].isna()
                | (reprojected["InfestSqM"] <= MIN_GEOFALLBACK_AREA)
            ),
            reprojected["geom_area_sqm"],
        )
    else:
        reprojected["area_sqm"] = reprojected["geom_area_sqm"]

    print("\nArea diagnostics (first 5):")
    print(
        reprojected[["global_id", "InfestSqM", "geom_area_sqm", "area_sqm"]]
        .head()
        .to_string(index=False)
    )
    print("area_sqm stats ->", reprojected["area_sqm"].describe())

    # Build experimental datasets
    training_datasets = {}
    print("\n--- Generating Experimental Datasets ---")
    all_ctrl_rows = reprojected[reprojected["is_cheatgrass"] == 0]
    if include_controls and all_ctrl_rows.empty:
        print(
            "⚠ include_controls=True but no controls found in this file "
            "(is_cheatgrass==0). Proceeding with positives only."
        )

    for pct in percent_thresholds:
        for area in area_thresholds:
            name = f"percent_{pct}_area_{area}"

            pos_mask = (
                (reprojected["is_cheatgrass"] == 1)
                & (reprojected["primary_sp_percent"] >= pct)
                & (reprojected["area_sqm"] >= area)
            )
            pos_subset = reprojected[pos_mask]

            if include_controls and not all_ctrl_rows.empty:
                ctrl_subset = all_ctrl_rows[all_ctrl_rows["area_sqm"] >= area]
                combined = pd.concat([pos_subset, ctrl_subset], ignore_index=True)
            else:
                ctrl_subset = reprojected.iloc[0:0].copy()  # empty like reprojected
                combined = pos_subset.copy()

            print(f"\n>> {name}")
            print(f"  - Pos (cheatgrass) count: {len(pos_subset)}")
            print(f"  - Ctrl count: {len(ctrl_subset)}")
            if len(reprojected):
                print(
                    f"  - Pos keep %: {100 * len(pos_subset) / len(reprojected):.1f}% | "
                    f"Ctrl keep %: {100 * len(ctrl_subset) / len(reprojected):.1f}%"
                )

            if len(pos_subset) == 0 and len(ctrl_subset) == 0:
                failing = reprojected[(reprojected["is_cheatgrass"] == 1)]
                if not failing.empty:
                    print("    Diagnostics (positive class candidates):")
                    print(
                        failing[["global_id", "primary_sp_percent", "area_sqm"]].head()
                    )

            training_datasets[name] = combined.reset_index(drop=True)

    print("\n✅ Dataset generation complete.")
    return training_datasets, final_test_set


# ------------------ NEW: BUILD DATASET WITH VALIDITY MASK -----------------------
# This does NOT touch the GEOJSON. It operates on your existing npy dataset
# (e.g. cheatgrass_data_quality_filtered_split) and makes a COPY with an extra
# *_valid.npy file per sample.
#
# - data shape assumed: (T, H, W, C)  float32
# - mask shape:          (T, H, W) or (H, W)
# - valid mask:          (T, H, W)  uint8 (1=observed, 0=missing/imputed)
#
# We treat a pixel as "valid" if:
#   * all bands are finite (no NaNs / infs), AND
#   * at least one band is non-zero (sum |band| > 0)
#
# Configure source & destination roots with env vars if you like:
#   CHEATGRASS_SRC_ROOT
#   CHEATGRASS_VALID_ROOT
# ------------------------------------------------------------------------------
DEFAULT_SRC_ROOT = "/home/rbielski/SAL_Git_Projects/Cheatgrass/cheatgrass_data_quality_filtered_split"
DEFAULT_VALID_ROOT = "/home/rbielski/SAL_Git_Projects/Cheatgrass/cheatgrass_data_validmask_copy"

def build_validmask_dataset(
    src_root: str | Path = None,
    dst_root: str | Path = None,
) -> None:
    """Clone existing *_data.npy / *_mask.npy tree and add *_valid.npy files.

    Nothing is overwritten in src_root; everything is written under dst_root.
    """
    src_root = Path(os.getenv("CHEATGRASS_SRC_ROOT", src_root or DEFAULT_SRC_ROOT))
    dst_root = Path(os.getenv("CHEATGRASS_VALID_ROOT", dst_root or DEFAULT_VALID_ROOT))

    print(f"\n--- Building validity-mask dataset ---")
    print(f"Source root:      {src_root}")
    print(f"Destination root: {dst_root}")
    dst_root.mkdir(parents=True, exist_ok=True)

    if not src_root.exists():
        raise FileNotFoundError(f"Source root not found: {src_root}")

    # walk splits like train / val / test (or any subdirs)
    n_samples = 0
    for split_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        split_name = split_dir.name
        out_split = dst_root / split_name
        out_split.mkdir(parents=True, exist_ok=True)
        print(f"\n[split: {split_name}]")

        data_files = sorted(split_dir.glob("*_data.npy"))
        if not data_files:
            print("  (no *_data.npy files found)")
            continue

        for data_path in data_files:
            base = data_path.stem.replace("_data", "")
            mask_path = data_path.with_name(f"{base}_mask.npy")
            if not mask_path.exists():
                print(f"  ⚠ skipping {base}: no mask file found")
                continue

            # Load data (T,H,W,C) and mask
            data = np.load(data_path).astype(np.float32)
            mask = np.load(mask_path)

            if data.ndim != 4:
                raise ValueError(f"{data_path.name}: expected data shape (T,H,W,C), got {data.shape}")

            T, H, W, C = data.shape

            # Valid where all bands finite AND not all bands zero
            finite = np.isfinite(data).all(axis=-1)            # (T,H,W)
            nonzero = (np.abs(data).sum(axis=-1) > 0.0)        # (T,H,W)
            valid = (finite & nonzero).astype(np.uint8)        # (T,H,W)

            # Write copies into new tree
            out_data = out_split / f"{base}_data.npy"
            out_mask = out_split / f"{base}_mask.npy"
            out_valid = out_split / f"{base}_valid.npy"

            np.save(out_data, data)
            np.save(out_mask, mask)
            np.save(out_valid, valid)

            n_samples += 1

        print(f"  -> wrote {n_samples} samples (cumulative)")

    print(f"\n✅ Done. Total samples with validity masks: {n_samples}")
    print(f"   New dataset root: {dst_root}")


# -------------------------------- EXECUTION -------------------------------------
if __name__ == "__main__":
    # 1) Build experimental geojson-based datasets (unchanged behaviour)
    experimental_sets, final_test_set = generate_experimental_datasets(
        filepath=INPUT_PATH,
        percent_thresholds=PERCENT_VARIATIONS,
        area_thresholds=AREA_VARIATIONS,
        target_crs=TARGET_CRS,
        include_controls=False,  # your file appears to be all cheatgrass
    )

    print(f"\nSummary of generated experimental sets: {len(experimental_sets)} total")
    if experimental_sets:
        first_key = next(iter(experimental_sets.keys()))
        sample_df = experimental_sets[first_key]
        print(
            f"Example set '{first_key}' rows: {len(sample_df)} | "
            f"Columns: {list(sample_df.columns)[:12]} ..."
        )

    # 2) OPTIONAL: build cloned npy dataset with validity masks.
    #    Uncomment the next line when you're ready to run it, or call
    #    build_validmask_dataset() from a notebook.
    #
    build_validmask_dataset()


Input dataset: /home/rbielski/SAL_Git_Projects/Cheatgrass/filtered_cheatgrass_cleaned_valid.geojson (override with CHEATGRASS_INPUT_PATH)

--- Loading Cleaned Cheatgrass Dataset ---
✅ Loaded 6446 rows | 91 columns
✅ Geometry-valid rows: 6446
Alias match positives (is_cheatgrass==1): 6446 / 6446
  global_id       primary_sp  primary_sp_percent  InfestSqM  is_cheatgrass
0   5072578  BROMUS TECTORUM                 3.0        NaN              1
1   5072579  BROMUS TECTORUM                 3.0        NaN              1
2   5072581  BROMUS TECTORUM                 3.0        NaN              1
3   5072582  BROMUS TECTORUM                 3.0        NaN              1
4   5072583  BROMUS TECTORUM                 3.0        NaN              1
[BASE] class balance is_cheatgrass -> {1: 6446}
[BASE] primary_sp_percent stats -> min=0.0 max=100.0
[BASE] InfestSqM summary -> count>0=1589 zeros/NaN=4857
⚠ Not enough class diversity or too few rows for stratified split; using random split.

--- Maste